# 1.环境准备 + 生成数据

In [40]:
import numpy as np
import torch
from torch import nn
from torch.utils import data
import matplotlib.pyplot as plt
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

# 2.DataLoader 读取数据（替代手写 data_iter）

In [41]:
def load_array(data_arrays, batch_size, is_train = True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)    # 把 (X, y) 打包成"一对一配对"的数据集
    return data.DataLoader(dataset, batch_size, shuffle = is_train)
batch_size = 10
data_iter = load_array((features, labels), batch_size)
'''
TensorDataset(*data_arrays)：* 是把 (features, labels) 这个元组"拆开"成两个参数传进去，让 X 和 y 按位置一一配对。以后你取第 i 个样本，X[i] 和 y[i] 自动对应。

DataLoader：内部帮你做了打乱 + 按 batch_size 切片 + 批量迭代——这三件事手写版全是自己写的。

shuffle=is_train：训练要打乱（True），测试不打乱（False）。
'''

# 3.用 nn.Sequential 定义模型

In [42]:
net = nn.Sequential(nn.Linear(2,1))  # 输入2维 → 输出1维
'''
nn.Linear(2, 1) 是一个模块（Module），它内部自动创建了参数：
weight：形状 (1, 2)（输出维度 × 输入维度）
bias：形状 (1,)
并且默认就带 requires_grad=True
nn.Sequential(...) 是一个容器，把层按顺序排起来，方便以后叠更多层（这正是为 MLP/CNN 打基础的地方）。
调 net(X) 就等价于手写版的 linreg(X, w, b)，自动完成 X@w + b 的前向计算。
'''

# 4.初始化模型参数

In [43]:
net[0].weight.data.normal_(0, 0.01)    # 用正态分布 N(0,0.01) 填满 weight
net[0].bias.data.fill_(0)             # b = 0,bias 全填 0
'''
net[0]：Sequential 是有序容器，[0] 就是第 0 层，也就是那个 nn.Linear(2,1)。
.weight / .bias：拿到这一层的参数。
.data：直接绕过 autograd 计算图去改数值。因为初始化不是"学习"，不需要记录梯度，所以动 .data 而不是动参数本身。
这就是手写版 w = torch.normal(0, 0.01, size=(2,1)) ,完全一样，只是换了写法：手写版在创建时初始化，简洁版在创建后覆盖。
'''

tensor([0.])

# 5.定义损失函数 + 优化器

In [44]:
loss = nn.MSELoss()   # 均方误差（默认reduce成标量, 不用手动sum）
trainer = torch.optim.SGD(net.parameters(), lr = 0.03)
# loss = nn.MSELoss()   # 内部是 mean( (y_hat - y)**2 )
'''
nn.MSELoss() 默认 reduction='mean'，直接给你一个标量，所以 l.backward() 直接用。

net.parameters()：返回容器里所有可训练参数的列表（这里就是 weight、bias 两个）。优化器拿到它，就知道"该更新谁"了。
optim.SGD(net.parameters(), lr=0.03) 里的第一个参数就是 net.parameters()——它把"该更新谁"这件事交出去了，手写版里你手动传 [w, b]，这里传的是 net 内部自动管理的参数。
训练时，简洁版把 sgd 里那两件事拆成了两个 API 调用：
trainer.zero_grad() → 对应手写版 param.grad.zero_()
trainer.step() → 对应手写版 param -= lr * param.grad / batch_size
注意一个隐藏细节：手写版里有 / batch_size，简洁版里没有。为什么？
手写版用 l.sum()（求和）算梯度 → 梯度是"一批之和" → 所以要除以 batch_size 变成"平均"；
简洁版用 MSELoss（mean）算梯度 → 梯度本来就是平均 → 不用再除。
'''

# 6. 训练主循环（3 个 epoch）

In [45]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:             # 取一批数据
        l = loss(net(X), y)            # 前向 + 计算损失(标量)
        trainer.zero_grad()            # 等价于手写版的 grad.zero_()
        l.backward()                   # 反向传播
        trainer.step()                 # 等价于手写版 sgd 的参数更新
    l = loss(net(features), labels)    # 整个数据集上的损失
    print(f'epoch{epoch + 1}, loss{l:f}')
'''    
data_iter(...)（调用生成器函数）; data_iter（直接迭代 DataLoader）
net(X, w, b)（手动传参）; net(X)（参数在 net 内部）
藏在 sgd 末尾 ; 显式 trainer.zero_grad() 写在开头
l.sum().backward() ; l.backward()
sgd([w,b],...)（手动） ; trainer.step()（自动）
'''

epoch1, loss0.000235
epoch2, loss0.000093
epoch3, loss0.000091


# 7.验证学到的参数

In [46]:
w = net[0].weight.data
print('w 的估计误差 :', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b 的估计误差:',true_b - b)

w 的估计误差 : tensor([ 0.0010, -0.0004])
b 的估计误差: tensor([-0.0004])


# 8.小结

| 功能 | 手写版（Day 1） | 简洁版（3.3） | 关键区别 |
| ---- | ---- | ---- | ---- |
| 数据迭代 | 手写 data_iter(batch_size) 生成器 + yield | DataLoader + TensorDataset | 简洁版自带 shuffle、自动分批 |
| 模型 | 手写 linreg(X, w, b) = X@w + b | nn.Sequential(nn.Linear(2,1)) | 简洁版自动管理参数、前向 |
| 参数初始化 | 手写 w = normal(0,0.01) + requires_grad_() | net[0].weight.data.normal_() | 简洁版默认已带 requires_grad=True |
| 损失 | 手写 squared_loss + l.sum().backward() | nn.MSELoss() | 简洁版自动 reduce 成标量，无需手动 sum |
| 优化 | 手写 sgd(params, lr, batch_size) + grad.zero_() | optim.SGD + trainer.zero_grad() + trainer.step() | 简洁版封装了梯度清零+更新 |
| 梯度更新 | with torch.no_grad(): param -= lr*grad | trainer.step() | 简洁版自动在 no_grad 下更新 |